# Kaggle — 11 Aday Ham Log İncelemesi (No-Inference)

Bu notebook, PR #222 dalındaki sabitlenmiş kanıt ZIP'ini indirir, SHA-256 ve ZIP bütünlüğünü doğrular, başarısız 11 adayın ilgili ham loglarını envanterler ve bulguları GitHub'a yükler.

**Güvenlik sınırı:** Model yükleme, model çıkarımı, eğitim, kod yaması uygulama veya aday kabul kararı yoktur. Otomatik etiketler yalnızca ön sınıflandırmadır; nihai değerlendirme değildir.

**Kaggle ayarları:** Internet = On, GPU = Off. Kaggle Secrets içinde `GITHUB_TOKEN` adlı, yalnızca bu depoda Contents read/write yetkili token bulunmalı.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, json, csv, hashlib, zipfile, io, re, base64, requests

REPO = 'MahmutVezir/dijitalikiz'
BRANCH = 'research/2026-11-16-followup-evidence-validation'
BASE_PATH = 'research_candidates/evidence_abstention/kaggle_runs/2026-11-17_b4b8b536e82e'
ZIP_NAME = 'kaggle_no_inference_candidate_evidence.zip'
SUMMARY_NAME = 'kaggle_candidate_job_summary.json'
EXPECTED_ZIP_SHA256 = 'b4b8b536e82e7ac93e94bbb51d63f71185a2b7ab2515178480189f8cf783b777'
EXPECTED_SUMMARY_SHA256 = 'd0bba8618d4af349ae20348ba0a8b046fbfe0d1cc2e81e777ab3bdb27bfd564a'
OUT = Path('/kaggle/working/no_inference_log_review')
OUT.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
print('Run ID:', RUN_ID)
print('Output:', OUT)


In [ ]:
# GitHub secret'i güvenli biçimde al; token hiçbir çıktıya yazdırılmaz.
try:
    from kaggle_secrets import UserSecretsClient
    GITHUB_TOKEN = UserSecretsClient().get_secret('GITHUB_TOKEN')
except Exception as exc:
    raise RuntimeError('Kaggle Secrets içinde GITHUB_TOKEN bulunamadı veya okunamadı.') from exc
if not GITHUB_TOKEN:
    raise RuntimeError('GITHUB_TOKEN boş.')
HEADERS = {'Authorization': f'Bearer {GITHUB_TOKEN}', 'Accept': 'application/vnd.github+json', 'X-GitHub-Api-Version': '2022-11-28'}

def download_repo_file(filename, expected_sha):
    url = f'https://raw.githubusercontent.com/{REPO}/{BRANCH}/{BASE_PATH}/{filename}'
    response = requests.get(url, timeout=180)
    if response.status_code != 200:
        api_url = f'https://api.github.com/repos/{REPO}/contents/{BASE_PATH}/{filename}?ref={BRANCH}'
        response = requests.get(api_url, headers={**HEADERS, 'Accept':'application/vnd.github.raw'}, timeout=180)
    response.raise_for_status()
    data = response.content
    actual = hashlib.sha256(data).hexdigest()
    if actual != expected_sha:
        raise ValueError(f'{filename} SHA-256 uyuşmuyor: {actual}')
    print(f'{filename}: {len(data):,} bytes | SHA-256 doğrulandı')
    return data

summary_bytes = download_repo_file(SUMMARY_NAME, EXPECTED_SUMMARY_SHA256)
zip_bytes = download_repo_file(ZIP_NAME, EXPECTED_ZIP_SHA256)
summary = json.loads(summary_bytes.decode('utf-8'))
assert summary.get('model_loaded') is False
assert summary.get('model_inference_performed') is False
assert summary.get('training_performed') is False
print('Summary jobs:', summary.get('completed_job_count'), '/', summary.get('expected_job_count'))
print('Model/inference/training flags: all false')


In [ ]:
# ZIP CRC bütünlüğü + dosya envanteri. İçeriği çalıştırmaz; yalnızca okur.
zf = zipfile.ZipFile(io.BytesIO(zip_bytes))
bad_member = zf.testzip()
if bad_member is not None:
    raise RuntimeError(f'ZIP CRC hatası: {bad_member}')
members = [m for m in zf.infolist() if not m.is_dir()]
print('ZIP integrity: PASS')
print('Archive files:', len(members))
archive_names = {m.filename for m in members}
print('Top-level folders:', sorted({n.split('/')[0] for n in archive_names if '/' in n}))


In [ ]:
# Başarısız aday job'larını summary'den al. Preflight ve başarılı Dishka job'u dışarıda.
failed_jobs = [j for j in summary.get('jobs', []) if j.get('outcome') == 'failure']
assert len(failed_jobs) == 11, f'Beklenen 11 başarısız aday job; bulunan: {len(failed_jobs)}'

def safe_read(member_name, limit=2_000_000):
    try:
        info = zf.getinfo(member_name)
        if info.file_size > limit:
            return f'[SKIPPED: {info.file_size} bytes exceeds read limit]'
        return zf.read(member_name).decode('utf-8', errors='replace')
    except KeyError:
        return '[ARCHIVE_MEMBER_MISSING]'

def classify(job, texts):
    failed = '; '.join(job.get('failed_steps') or [])
    blob = ('\n'.join(texts)).lower()
    if any(k in blob for k in ['could not resolve host', 'connection timed out', 'connection reset', 'network is unreachable', 'failed to connect']):
        category = 'network_or_remote_access'
    elif any(k in blob for k in ['no matching distribution found', 'modulenotfounderror', 'no module named', 'dependency', 'could not build wheels']):
        category = 'dependency_or_environment'
    elif any(k in blob for k in ['permission denied', 'authentication failed', 'repository not found', 'could not read from remote repository']):
        category = 'git_auth_or_repository_access'
    elif any(k in blob for k in ['patch failed', 'hunk failed', 'does not apply', 'already exists in working directory']):
        category = 'patch_or_source_mismatch'
    elif any(k in blob for k in ['assertionerror', 'failed (', 'error: ', 'failed:']):
        category = 'test_or_reproduction_failure_requires_review'
    elif 'clone' in failed.lower() or 'patch' in failed.lower():
        category = 'source_checkout_or_patch_step_failed_cause_unresolved'
    elif 'dataset' in failed.lower() or 'test' in failed.lower():
        category = 'test_step_failed_cause_unresolved'
    else:
        category = 'unclassified_inspect_raw_logs'
    return category

records = []
for job in failed_jobs:
    job_name = job.get('job', 'unknown-job')
    artifact_files = []
    for artifact in job.get('artifacts', []):
        artifact_files.extend(artifact.get('files', []))
    available = [p for p in artifact_files if p in archive_names]
    missing = [p for p in artifact_files if p not in archive_names]
    relevant = [p for p in available if p.lower().endswith(('.log','.txt','.json','.yaml','.yml','.patch','.diff'))]
    texts = [safe_read(p) for p in relevant]
    failed_steps = job.get('failed_steps') or []
    skipped = [s.get('name') for s in job.get('steps',[]) if 'skipped' in str(s.get('status',''))]
    records.append({
        'job': job_name,
        'outcome': job.get('outcome'),
        'duration_seconds': job.get('duration_seconds'),
        'failed_steps': failed_steps,
        'preliminary_category': classify(job, texts),
        'artifact_declared_count': len(artifact_files),
        'artifact_found_count': len(available),
        'artifact_missing_count': len(missing),
        'missing_artifacts': missing,
        'relevant_files': relevant,
        'skipped_steps': skipped,
        'log_excerpt_candidates': []
    })
    # Kısa hata bağlamları: satır numarası ve ham metin parçası; yorum yerine kanıt.
    excerpts = []
    for p, content in zip(relevant, texts):
        lines = content.splitlines()
        hits = [i for i,line in enumerate(lines) if re.search(r'(?i)(error|exception|traceback|fatal|failed|not found|denied|timed out|could not)', line)]
        for i in hits[:12]:
            lo, hi = max(0,i-2), min(len(lines),i+3)
            excerpts.append({'file':p, 'line':i+1, 'context':'\n'.join(lines[lo:hi])[:1600]})
    records[-1]['log_excerpt_candidates'] = excerpts[:60]

print('Failure records:', len(records))
for r in records:
    print('\n', r['job'], '|', r['preliminary_category'])
    print('Failed:', r['failed_steps'])
    print('Artifacts found/missing:', r['artifact_found_count'], '/', r['artifact_missing_count'])
    print('Skipped:', r['skipped_steps'])


In [ ]:
# Makinece okunabilir ve insan incelemesine uygun rapor üret.
report = {
    'schema_version': '1.0',
    'run_id_utc': RUN_ID,
    'repository': REPO,
    'source_branch': BRANCH,
    'source_archive_path': f'{BASE_PATH}/{ZIP_NAME}',
    'source_archive_sha256': EXPECTED_ZIP_SHA256,
    'source_summary_sha256': EXPECTED_SUMMARY_SHA256,
    'zip_integrity': 'pass',
    'expected_failed_candidate_jobs': 11,
    'observed_failed_candidate_jobs': len(records),
    'model_loaded': False,
    'model_inference_performed': False,
    'training_performed': False,
    'acceptance_decisions_made': False,
    'interpretation': 'Categories are deterministic preliminary triage labels only. A failed job does not establish candidate invalidity. Inspect raw logs and rerun after fixing environment/source issues.',
    'candidate_reviews': records
}
json_path = OUT / 'candidate_log_review.json'
json_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')

csv_path = OUT / 'candidate_failure_inventory.csv'
with csv_path.open('w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=['job','outcome','failed_steps','preliminary_category','artifact_declared_count','artifact_found_count','artifact_missing_count','missing_artifacts','relevant_files','skipped_steps'])
    writer.writeheader()
    for r in records:
        row = {k:r.get(k) for k in writer.fieldnames}
        for k in ['failed_steps','missing_artifacts','relevant_files','skipped_steps']:
            row[k] = json.dumps(row[k], ensure_ascii=False)
        writer.writerow(row)

md = [
    '# 11 Aday Kaggle Log İncelemesi', '',
    f'- Run ID (UTC): `{RUN_ID}`',
    f'- Kaynak ZIP SHA-256: `{EXPECTED_ZIP_SHA256}`',
    '- ZIP CRC bütünlüğü: **PASS**',
    '- Model yükleme/çıkarım/eğitim: **YOK**',
    '- Otomatik vaka kabul kararı: **YOK**', '',
    '> Ön sınıflandırma etiketleri yalnızca log taramasına dayalı triyajdır. Başarısız job tek başına adayın geçersiz olduğunu göstermez.', '',
    '| Aday job | Başarısız adım | Ön kategori | Bulunan/eksik artifact |',
    '|---|---|---|---|'
]
for r in records:
    md.append(f"| `{r['job']}` | {', '.join(r['failed_steps']) or '—'} | `{r['preliminary_category']}` | {r['artifact_found_count']}/{r['artifact_missing_count']} |")
for r in records:
    md += ['', f"## {r['job']}", '', f"- Başarısız adım: {', '.join(r['failed_steps']) or '—'}", f"- Atlanan adımlar: {', '.join(r['skipped_steps']) or '—'}", f"- Artifact: {r['artifact_found_count']} bulundu; {r['artifact_missing_count']} eksik", f"- Ön kategori: `{r['preliminary_category']}`", '']
    if r['missing_artifacts']:
        md += ['Eksik artifact yolları:'] + [f'- `{p}`' for p in r['missing_artifacts']]
    for e in r['log_excerpt_candidates'][:12]:
        md += [f"**{e['file']}:{e['line']}**", '```text', e['context'], '```']
md_path = OUT / 'candidate_log_review.md'
md_path.write_text('\n'.join(md) + '\n', encoding='utf-8')
print('Created:', json_path, csv_path, md_path)


In [ ]:
# Raporları GitHub'a yükle. Token hiçbir zaman ekrana basılmaz veya dosyaya yazılmaz.
api_base = f'https://api.github.com/repos/{REPO}/contents'
def upload_text_file(local_path, repo_path):
    raw = local_path.read_bytes()
    url = f'{api_base}/{repo_path}'
    existing = requests.get(url, headers=HEADERS, params={'ref': BRANCH}, timeout=60)
    sha = existing.json().get('sha') if existing.status_code == 200 else None
    if existing.status_code not in (200,404):
        raise RuntimeError(f'GitHub mevcut dosya kontrolü başarısız: HTTP {existing.status_code}')
    payload = {
        'message': f'Add Kaggle candidate log review {RUN_ID}',
        'content': base64.b64encode(raw).decode('ascii'),
        'branch': BRANCH
    }
    if sha:
        payload['sha'] = sha
    response = requests.put(url, headers=HEADERS, json=payload, timeout=120)
    if response.status_code not in (200,201):
        raise RuntimeError(f'GitHub upload failed HTTP {response.status_code}: {response.text[:1000]}')
    return response.json()['content']['html_url'], response.json()['commit']['sha']

dest_dir = f'research_candidates/evidence_abstention/kaggle_runs/log_review_{RUN_ID}'
uploaded = []
for local, name in [(json_path,'candidate_log_review.json'),(csv_path,'candidate_failure_inventory.csv'),(md_path,'candidate_log_review.md')]:
    url, commit = upload_text_file(local, f'{dest_dir}/{name}')
    uploaded.append({'file':name,'url':url,'commit':commit,'bytes':local.stat().st_size,'sha256':hashlib.sha256(local.read_bytes()).hexdigest()})
print(json.dumps({'upload_status':'SUCCESS','run_id':RUN_ID,'files':uploaded}, ensure_ascii=False, indent=2))


## Çalıştırma sonrası kontrol

Son hücrede `upload_status: SUCCESS` görünmeli. GitHub'da `research_candidates/evidence_abstention/kaggle_runs/log_review_<RUN_ID>/` klasöründe üç rapor dosyası oluşur. Çıktıdaki GitHub bağlantılarını buraya gönder; raporları ve log alıntılarını inceleyip her aday için kesin sınıflandırmayı yapacağım. Notebook, ön sınıflandırma sonuçlarını nihai bilimsel karar olarak sunmaz.
